# ETL Pipeline Demonstration

This notebook demonstrates a simple ETL pipeline using PySpark and Delta Lake to process data through Bronze, Silver, and Gold layers in a MinIO-based Data Lakehouse.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg

## Initialize Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName("DataLakehouseDemo") \
    .getOrCreate()

## Bronze Layer: Ingest Raw Data

In [ ]:
data = [("Alice", 25, "Engineer"),
        ("Bob", 30, "Doctor"),
        ("Charlie", 35, "Artist"),
        ("David", 40, "Engineer"),
        ("Eve", 45, "Doctor")]
columns = ["name", "age", "occupation"]
df = spark.createDataFrame(data, columns)

bronze_path = "s3a://datalakehouse/bronze/raw_data"
df.write.mode("overwrite").csv(bronze_path, header=True)

## Silver Layer: Cleaned and Transformed Data

In [ ]:
bronze_df = spark.read.csv(bronze_path, header=True, inferSchema=True)

# Simple transformation: filter for engineers
silver_df = bronze_df.filter(col("occupation") == "Engineer")

silver_path = "s3a://datalakehouse/silver/engineers"
silver_df.write.format("delta").mode("overwrite").save(silver_path)

## Gold Layer: Aggregated and Business-Ready Data

In [ ]:
silver_delta_df = spark.read.format("delta").load(silver_path)

# Aggregation: calculate average age of engineers
gold_df = silver_delta_df.agg(avg("age").alias("average_age"))

gold_path = "s3a://datalakehouse/gold/avg_engineer_age"
gold_df.write.format("delta").mode("overwrite").save(gold_path)

## Verification

In [ ]:
print("Reading from Gold Layer:")
spark.read.format("delta").load(gold_path).show()